# Segmentation using background subtraction

Use background substraction methods to properly segment the moving objects from their background. Use one of the videos with static camera.

Use the following approaches:
- Accumulated weighted image
- Mixture of Gaussian (MOG2)


In [30]:
import cv2 as cv
import numpy as np

In [31]:
# Video file names
file1 = 'data/crosswalk.avi'
file2 = 'data/fourway.avi'
file3 = 'data/night.avi'

### Accumulated weighted image

In [32]:
def accumulate_weighted_image(file: str, alpha: float = 0.02, scale: float = 0.5, convert_to_gray: bool = True):
    # Load the video
    cap = cv.VideoCapture(file)

    # Read the first frame
    _, frame = cap.read()

    # Convert the first frame to grayscale
    if convert_to_gray:
        frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Initialize the background model
    bg_model = np.float32(frame)

    while True:
        # Read the next frame of the video
        success, frame = cap.read()
        # Exit the loop if the frame was not successfully read
        if not success:
            cv.waitKey(0)
            break

        # Convert the current frame to grayscale
        if convert_to_gray:
            frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Update the background model using an accumulated weighted image
        cv.accumulateWeighted(frame, bg_model, alpha)

        # Subtract the background
        background = cv.convertScaleAbs(bg_model)
        diff = cv.absdiff(frame, background)
        _, thresh = cv.threshold(diff, 25, 255, cv.THRESH_BINARY)

        if scale != 1.0:
            frame = cv.resize(frame, (0, 0), fx=scale, fy=scale)
            background = cv.resize(background, (0, 0), fx=scale, fy=scale)
            thresh = cv.resize(thresh, (0, 0), fx=scale, fy=scale)

        # Display the resulting images
        cv.imshow("Frame", frame)
        cv.imshow("Thresh", thresh)
        cv.imshow("Background", background)

        # Exit the program if the user presses the 'q' key
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the VideoCapture and close all windows
    cap.release()
    cv.destroyAllWindows()

In [33]:
accumulate_weighted_image(file1, alpha=0.02, scale=0.5, convert_to_gray=True)

### Mixture of Gaussian (MOG2)

In [34]:
def background_subtractor(file: str, subtractor: str, scale: float = 0.5, apply_morph: bool = True):
    # capture frames from a camera
    cap = cv.VideoCapture(file)

    # create kernel for morphological operation
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3))

    # creating object
    subtractor = subtractor.upper()
    if subtractor == 'MOG':
        fgbg = cv.bgsegm.createBackgroundSubtractorMOG()
    elif subtractor == 'MOG2':
        fgbg = cv.createBackgroundSubtractorMOG2()
    elif subtractor == 'GMG':
        fgbg = cv.bgsegm.createBackgroundSubtractorGMG()
    elif subtractor == 'KNN':
        fgbg = cv.createBackgroundSubtractorKNN()
    else:
        raise ValueError('Unknown background subtractor')

    # read the frames from the camera
    while True:
        # read frames
        ret, frame = cap.read()

        if not ret:
            cv.waitKey(0)
            break

        img = cv.resize(frame, (0, 0), fx=scale, fy=scale)

        # apply mask for background subtraction
        fg_mask = fgbg.apply(img)

        # morphological operation
        if apply_morph:
            fg_mask = cv.morphologyEx(fg_mask, cv.MORPH_OPEN, kernel)

        cv.imshow('Original', img)
        cv.imshow(f'FG Mask ({subtractor})', fg_mask)

        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv.destroyAllWindows()


def all_background_subtractors(file: str, scale: float = 0.5, apply_morph: bool = True):
    # capture frames from a camera
    cap = cv.VideoCapture(file)

    # create kernel for morphological operation
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3))

    # creating object
    fgbg_mog = cv.bgsegm.createBackgroundSubtractorMOG()
    fgbg_mog2 = cv.createBackgroundSubtractorMOG2()
    fgbg_gmg = cv.bgsegm.createBackgroundSubtractorGMG()
    fgbg_knn = cv.createBackgroundSubtractorKNN()

    # read the frames from the camera
    while True:
        # read frames
        ret, frame = cap.read()

        if not ret:
            cv.waitKey(0)
            break

        img = cv.resize(frame, (0, 0), fx=scale, fy=scale)

        # apply mask for background subtraction
        fg_mask_mog = fgbg_mog.apply(img)
        fg_mask_mog2 = fgbg_mog2.apply(img)
        fg_mask_gmg = fgbg_gmg.apply(img)
        fg_mask_knn = fgbg_knn.apply(img)

        # morphological operation
        if apply_morph:
            fg_mask_mog_morph = cv.morphologyEx(fg_mask_mog, cv.MORPH_OPEN, kernel)
            fg_mask_mog2_morph = cv.morphologyEx(fg_mask_mog2, cv.MORPH_OPEN, kernel)
            fg_mask_gmg_morph = cv.morphologyEx(fg_mask_gmg, cv.MORPH_OPEN, kernel)
            fg_mask_knn_morph = cv.morphologyEx(fg_mask_knn, cv.MORPH_OPEN, kernel)

            cv.imshow('MOG Morph', fg_mask_mog_morph)
            cv.imshow('MOG2 Morph', fg_mask_mog2_morph)
            cv.imshow('GMG Morph', fg_mask_gmg_morph)
            cv.imshow('KNN Morph', fg_mask_knn_morph)

        cv.imshow('Original', img)
        cv.imshow('MOG', fg_mask_mog)
        cv.imshow('MOG2', fg_mask_mog2)
        cv.imshow('GMG', fg_mask_gmg)
        cv.imshow('KNN', fg_mask_knn)

        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv.destroyAllWindows()

In [35]:
all_background_subtractors(file1, scale=0.4, apply_morph=True)

In [36]:
background_subtractor(file1, 'mog2', scale=0.4, apply_morph=True)